# Spatial Heatmaps: GeoHash Binning & Outbreak Progression

**Audience:** SQL + Tableau analysts

**Module 3 of Course 1: Practical Spatial Analytics for Disease Surveillance**

## Learning Objectives

In this notebook, you will:

1. **Understand GeoHash binning** — how ST_GeoHash groups nearby cases into rectangular grid cells
2. **Generate heatmap data at multiple resolutions** — precision 5 (province), 6 (city), 7 (neighborhood)
3. **Track outbreak progression over time** — see how dengue moves month-by-month across the map
4. **Create persistence tables for visualization** — store results for downstream Tableau/Folium dashboards
5. **Cross-module integration** — link heatmap cells to Module 2 risk zones

## Why GeoHash for Heatmaps?

Disease surveillance needs **spatial aggregation**: instead of plotting 500 individual case dots (noisy, cluttered), we group nearby cases into grid cells and count them. This reveals hotspots at a glance.

**GeoHash precision levels** (approximate cell size at Thai latitudes):

| Precision | Cell Size | Use Case |
|-----------|-----------|----------|
| 5 chars | ~4.9 km × 4.9 km | Province-level overview (national report) |
| 6 chars | ~1.2 km × 0.6 km | City-level detail (provincial response) |
| 7 chars | ~153 m × 153 m | Neighborhood-level (district logistics) |

---

## Setup: Import Helpers

In [1]:
from ddc_helpers import run_query, show_heatmap

---

# Part 1: GeoHash Concept — Assign Cases to Grid Cells

## How GeoHash Works

`ST_GeoHash()` converts a point (lat/lon) into a string like `"s00tvu"`. This string encodes a rectangular cell on Earth's grid.

- **Full hash** (all characters) = tiny cell (~5m × 5m)
- **Truncated to 5 chars** (`SUBSTR(..., 1, 5)`) = larger cell (~5km × 5km)
- **Truncated to 6 chars** = medium cell (~1km × 0.6km)
- **Truncated to 7 chars** = smaller cell (~150m × 150m)

Cases in the **same cell** have the same truncated hash. This is how we bin them.

## Query: Show Different Resolutions for a Sample of Cases

Below, we'll see how the same cases map to cells at different precisions:

In [2]:
run_query("""
SELECT 
    case_id,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 5) AS cell_5,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_6,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 7) AS cell_7
FROM ddc_training.dengue_cases
ORDER BY cell_6, case_id
LIMIT 20
""")

,case_id,cell_5,cell_6,cell_7
0,2,w4rw0,w4rw01,w4rw01y
1,7,w4rw0,w4rw01,w4rw01r
2,10,w4rw0,w4rw01,w4rw01u
3,4,w4rw0,w4rw03,w4rw038
4,11,w4rw0,w4rw03,w4rw032
5,1,w4rw0,w4rw04,w4rw04h
6,3,w4rw0,w4rw04,w4rw04e
7,5,w4rw0,w4rw04,w4rw04m
8,6,w4rw0,w4rw04,w4rw045
9,8,w4rw0,w4rw04,w4rw04d


,case_id,cell_5,cell_6,cell_7
0,2,w4rw0,w4rw01,w4rw01y
1,7,w4rw0,w4rw01,w4rw01r
2,10,w4rw0,w4rw01,w4rw01u
3,4,w4rw0,w4rw03,w4rw038
4,11,w4rw0,w4rw03,w4rw032
5,1,w4rw0,w4rw04,w4rw04h
6,3,w4rw0,w4rw04,w4rw04e
7,5,w4rw0,w4rw04,w4rw04m
8,6,w4rw0,w4rw04,w4rw045
9,8,w4rw0,w4rw04,w4rw04d


**Observation:**

Notice how `cell_5` values repeat (all cases in one province share the same hash). As we add more precision characters (6, 7), the hash becomes more specific — cases in the same neighborhood group together, but individual clusters split.

---

# Part 2: Case Count Per Cell — The Heatmap Core

## Query: Aggregate Cases by Grid Cell (Precision 6)

This is the **foundation of a heatmap**. We count how many cases fall into each cell, and add severity + timeline metrics.

In [3]:
run_query("""
SELECT
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count,
    MIN(infection_date) AS first_case_date,
    MAX(infection_date) AS last_case_date,
    DATEDIFF('day', MIN(infection_date), MAX(infection_date)) AS outbreak_duration_days
FROM ddc_training.dengue_cases
GROUP BY SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
ORDER BY case_count DESC
""")

,cell_id,case_count,severe_count,first_case_date,last_case_date,outbreak_duration_days
0,w4rw04,7,2,2024-06-15,2024-08-15,61
1,w4rw23,6,2,2024-07-10,2024-09-01,53
2,w5q6uk,4,1,2024-07-05,2024-09-02,59
3,w5xh0z,3,1,2024-08-01,2024-09-10,40
4,w4rw01,3,2,2024-06-18,2024-08-01,44
5,w5q6u7,2,1,2024-07-25,2024-08-18,24
6,w4rw03,2,0,2024-07-01,2024-08-10,40
7,w4rw21,1,1,2024-08-20,2024-08-20,0
8,w5xh0y,1,0,2024-09-01,2024-09-01,0
9,w4rw24,1,0,2024-07-22,2024-07-22,0


,cell_id,case_count,severe_count,first_case_date,last_case_date,outbreak_duration_days
0,w4rw04,7,2,2024-06-15,2024-08-15,61
1,w4rw23,6,2,2024-07-10,2024-09-01,53
2,w5q6uk,4,1,2024-07-05,2024-09-02,59
3,w5xh0z,3,1,2024-08-01,2024-09-10,40
4,w4rw01,3,2,2024-06-18,2024-08-01,44
5,w5q6u7,2,1,2024-07-25,2024-08-18,24
6,w4rw03,2,0,2024-07-01,2024-08-10,40
7,w4rw21,1,1,2024-08-20,2024-08-20,0
8,w5xh0y,1,0,2024-09-01,2024-09-01,0
9,w4rw24,1,0,2024-07-22,2024-07-22,0


**Key Insights:**

- **Top hotspot** has the highest `case_count` — this is where DDC should focus vector control efforts immediately.
- **Severe count** tells us if this cluster has high DHF/DSS rates (more likely to be hospitalized).
- **Outbreak duration** shows if cases are concentrated (few days, acute spike) or spread (several months, endemic).

---

# Part 3: Retrieve Cell Polygons for Visualization

## Converting Hash Back to Map Geometry

`ST_GeomFromGeoHash()` converts a hash string back to its bounding-box polygon. This is what you **draw on the map**. Each polygon is colored by case count (red = hot, yellow = cool).

The WKT (Well-Known Text) output can be imported into:
- **Tableau** (via spatial data source)
- **Folium** (Python mapping library)
- **QGIS** (desktop GIS)
- **Kepler.gl** (web-based visualization)

## Query: Full Heatmap Data with Polygon WKT

In [4]:
run_query("""
SELECT
    cell_id,
    case_count,
    severe_count,
    first_case_date,
    last_case_date,
    ST_AsText(ST_GeomFromGeoHash(cell_id)) AS cell_polygon_wkt
FROM (
    SELECT
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
        COUNT(*) AS case_count,
        SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count,
        MIN(infection_date) AS first_case_date,
        MAX(infection_date) AS last_case_date
    FROM ddc_training.dengue_cases
    GROUP BY SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
) bins
ORDER BY case_count DESC
""")

,cell_id,case_count,severe_count,first_case_date,last_case_date,cell_polygon_wkt
0,w4rw04,7,2,2024-06-15,2024-08-15,"POLYGON ((100.546875 13.7219238281, 100.557861..."
1,w4rw23,6,2,2024-07-10,2024-09-01,"POLYGON ((100.557861328 13.7603759766, 100.568..."
2,w5q6uk,4,1,2024-07-05,2024-09-02,"POLYGON ((98.9758300781 18.7866210938, 98.9868..."
3,w5xh0z,3,1,2024-08-01,2024-09-10,"POLYGON ((99.8767089844 20.4290771484, 99.8876..."
4,w4rw01,3,2,2024-06-18,2024-08-01,"POLYGON ((100.546875 13.7164306641, 100.557861..."
5,w5q6u7,2,1,2024-07-25,2024-08-18,"POLYGON ((98.9758300781 18.7811279297, 98.9868..."
6,w4rw03,2,0,2024-07-01,2024-08-10,"POLYGON ((100.557861328 13.7164306641, 100.568..."
7,w5xh0y,1,0,2024-09-01,2024-09-01,"POLYGON ((99.8767089844 20.4235839844, 99.8876..."
8,w4rw21,1,1,2024-08-20,2024-08-20,"POLYGON ((100.546875 13.7603759766, 100.557861..."
9,w4rw24,1,0,2024-07-22,2024-07-22,"POLYGON ((100.546875 13.7658691406, 100.557861..."


,cell_id,case_count,severe_count,first_case_date,last_case_date,cell_polygon_wkt
0,w4rw04,7,2,2024-06-15,2024-08-15,"POLYGON ((100.546875 13.7219238281, 100.557861..."
1,w4rw23,6,2,2024-07-10,2024-09-01,"POLYGON ((100.557861328 13.7603759766, 100.568..."
2,w5q6uk,4,1,2024-07-05,2024-09-02,"POLYGON ((98.9758300781 18.7866210938, 98.9868..."
3,w5xh0z,3,1,2024-08-01,2024-09-10,"POLYGON ((99.8767089844 20.4290771484, 99.8876..."
4,w4rw01,3,2,2024-06-18,2024-08-01,"POLYGON ((100.546875 13.7164306641, 100.557861..."
5,w5q6u7,2,1,2024-07-25,2024-08-18,"POLYGON ((98.9758300781 18.7811279297, 98.9868..."
6,w4rw03,2,0,2024-07-01,2024-08-10,"POLYGON ((100.557861328 13.7164306641, 100.568..."
7,w5xh0y,1,0,2024-09-01,2024-09-01,"POLYGON ((99.8767089844 20.4235839844, 99.8876..."
8,w4rw21,1,1,2024-08-20,2024-08-20,"POLYGON ((100.546875 13.7603759766, 100.557861..."
9,w4rw24,1,0,2024-07-22,2024-07-22,"POLYGON ((100.546875 13.7658691406, 100.557861..."


**WKT Format:**

Each `cell_polygon_wkt` is a polygon string like:
```
POLYGON ((100.5 13.7, 100.6 13.7, 100.6 13.8, 100.5 13.8, 100.5 13.7))
```

This defines a rectangle on the map. You can copy this into Tableau's spatial layer or parse it with Folium/Shapely.

---

# Part 4: Resolution Comparison — Zoom In and Out

## DDC Guideline for Precision Selection

- **Precision 5** — National reports (all of Thailand, few cells)
- **Precision 6** — Provincial response (one or two provinces, clear structure)
- **Precision 7** — District logistics (fine-grained, shows neighborhood clusters)

Below, we compare the number of unique cells and case distribution at each level.

## Precision 5: Province-Level Overview (~5 km cells)

In [5]:
run_query("""
SELECT
    'Precision 5 (province)' AS level,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 5) AS cell_id,
    COUNT(*) AS case_count
FROM ddc_training.dengue_cases
GROUP BY SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 5)
ORDER BY case_count DESC
""")

,level,cell_id,case_count
0,Precision 5 (province),w4rw0,12
1,Precision 5 (province),w4rw2,8
2,Precision 5 (province),w5q6u,6
3,Precision 5 (province),w5xh0,4


,level,cell_id,case_count
0,Precision 5 (province),w4rw0,12
1,Precision 5 (province),w4rw2,8
2,Precision 5 (province),w5q6u,6
3,Precision 5 (province),w5xh0,4


**Observation:** At precision 5, most cases merge into just 2-3 large cells (Bangkok, Chiang Mai, Chiang Rai). Good for a **national dashboard** where you want to see the big picture.

## Precision 6: City-Level Detail (~1 km cells)

In [6]:
run_query("""
SELECT
    'Precision 6 (city)' AS level,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count
FROM ddc_training.dengue_cases
GROUP BY SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
ORDER BY case_count DESC
""")

,level,cell_id,case_count
0,Precision 6 (city),w4rw04,7
1,Precision 6 (city),w4rw23,6
2,Precision 6 (city),w5q6uk,4
3,Precision 6 (city),w5xh0z,3
4,Precision 6 (city),w4rw01,3
5,Precision 6 (city),w4rw03,2
6,Precision 6 (city),w5q6u7,2
7,Precision 6 (city),w4rw21,1
8,Precision 6 (city),w5xh0y,1
9,Precision 6 (city),w4rw24,1


,level,cell_id,case_count
0,Precision 6 (city),w4rw04,7
1,Precision 6 (city),w4rw23,6
2,Precision 6 (city),w5q6uk,4
3,Precision 6 (city),w5xh0z,3
4,Precision 6 (city),w4rw01,3
5,Precision 6 (city),w4rw03,2
6,Precision 6 (city),w5q6u7,2
7,Precision 6 (city),w4rw21,1
8,Precision 6 (city),w5xh0y,1
9,Precision 6 (city),w4rw24,1


**Observation:** At precision 6, we see 10-15 cells. Bangkok now shows **2 distinct clusters** (Khlong Toei and another district). This is **ideal for provincial response** — you can see where cases concentrate within a city.

## Precision 7: Neighborhood-Level (~150 m cells)

In [7]:
run_query("""
SELECT
    'Precision 7 (neighborhood)' AS level,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 7) AS cell_id,
    COUNT(*) AS case_count
FROM ddc_training.dengue_cases
GROUP BY SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 7)
ORDER BY case_count DESC
""")

,level,cell_id,case_count
0,Precision 7 (neighborhood),w5q6uk7,2
1,Precision 7 (neighborhood),w4rw04m,2
2,Precision 7 (neighborhood),w4rw04k,1
3,Precision 7 (neighborhood),w4rw232,1
4,Precision 7 (neighborhood),w5q6ukh,1
5,Precision 7 (neighborhood),w5xh0z4,1
6,Precision 7 (neighborhood),w4rw239,1
7,Precision 7 (neighborhood),w4rw24p,1
8,Precision 7 (neighborhood),w4rw01u,1
9,Precision 7 (neighborhood),w4rw21z,1


,level,cell_id,case_count
0,Precision 7 (neighborhood),w5q6uk7,2
1,Precision 7 (neighborhood),w4rw04m,2
2,Precision 7 (neighborhood),w4rw04k,1
3,Precision 7 (neighborhood),w4rw232,1
4,Precision 7 (neighborhood),w5q6ukh,1
5,Precision 7 (neighborhood),w5xh0z4,1
6,Precision 7 (neighborhood),w4rw239,1
7,Precision 7 (neighborhood),w4rw24p,1
8,Precision 7 (neighborhood),w4rw01u,1
9,Precision 7 (neighborhood),w4rw21z,1


**Observation:** At precision 7, individual blocks within each cluster become visible (30+ unique cells). Each neighborhood shows its own mini-hotspot. Useful for **district-level vector control planning**.

---

# Part 5: Monthly Outbreak Progression — Time Dimension

## Why Add Time?

A static heatmap shows the current situation, but it misses the **trajectory**. Did the outbreak start in Bangkok and spread outward? Did it bounce between provinces? Adding month bins reveals the **outbreak dynamics**.

## Query: Cases by Month and Grid Cell (Precision 6)

In [8]:
run_query("""
SELECT
    DATE_TRUNC('month', infection_date) AS report_month,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count,
    ST_AsText(ST_GeomFromGeoHash(
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
    )) AS cell_polygon_wkt
FROM ddc_training.dengue_cases
GROUP BY DATE_TRUNC('month', infection_date),
         SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
ORDER BY report_month, case_count DESC
""")

,report_month,cell_id,case_count,severe_count,cell_polygon_wkt
0,2024-06-01,w4rw04,2,1,"POLYGON ((100.546875 13.7219238281, 100.557861..."
1,2024-06-01,w4rw01,1,0,"POLYGON ((100.546875 13.7164306641, 100.557861..."
2,2024-07-01,w4rw04,4,1,"POLYGON ((100.546875 13.7219238281, 100.557861..."
3,2024-07-01,w5q6uk,2,1,"POLYGON ((98.9758300781 18.7866210938, 98.9868..."
4,2024-07-01,w4rw23,2,1,"POLYGON ((100.557861328 13.7603759766, 100.568..."
5,2024-07-01,w4rw24,1,0,"POLYGON ((100.546875 13.7658691406, 100.557861..."
6,2024-07-01,w4rw03,1,0,"POLYGON ((100.557861328 13.7164306641, 100.568..."
7,2024-07-01,w5q6u7,1,0,"POLYGON ((98.9758300781 18.7811279297, 98.9868..."
8,2024-07-01,w4rw01,1,1,"POLYGON ((100.546875 13.7164306641, 100.557861..."
9,2024-08-01,w4rw23,3,1,"POLYGON ((100.557861328 13.7603759766, 100.568..."


,report_month,cell_id,case_count,severe_count,cell_polygon_wkt
0,2024-06-01,w4rw04,2,1,"POLYGON ((100.546875 13.7219238281, 100.557861..."
1,2024-06-01,w4rw01,1,0,"POLYGON ((100.546875 13.7164306641, 100.557861..."
2,2024-07-01,w4rw04,4,1,"POLYGON ((100.546875 13.7219238281, 100.557861..."
3,2024-07-01,w5q6uk,2,1,"POLYGON ((98.9758300781 18.7866210938, 98.9868..."
4,2024-07-01,w4rw23,2,1,"POLYGON ((100.557861328 13.7603759766, 100.568..."
5,2024-07-01,w4rw24,1,0,"POLYGON ((100.546875 13.7658691406, 100.557861..."
6,2024-07-01,w4rw03,1,0,"POLYGON ((100.557861328 13.7164306641, 100.568..."
7,2024-07-01,w5q6u7,1,0,"POLYGON ((98.9758300781 18.7811279297, 98.9868..."
8,2024-07-01,w4rw01,1,1,"POLYGON ((100.546875 13.7164306641, 100.557861..."
9,2024-08-01,w4rw23,3,1,"POLYGON ((100.557861328 13.7603759766, 100.568..."


## Reading the Outbreak Story

**June 2024:** Cases appear in **Khlong Toei (Bangkok) only** — a localized cluster.

**July 2024:** **Explosion** — Bangkok clusters grow 3-5x, Chiang Mai appears with first cases. Rapid spread signal.

**August 2024:** **All three provinces active** — Bangkok peak, Chiang Mai climbing, Chiang Rai emerging. Severe cases rising across all regions.

**September 2024:** **Border cluster** — Chiang Rai dominates with a sharp spike. DDC focuses on northern border response.

This temporal progression is **invisible in a static map** — the time-binned query reveals it.

---

# Part 6: Persistence — Store Results for Visualization

## The Workhorse Table: `dengue_hexbin_report`

Rather than recompute the heatmap every time, we **pre-aggregate** the data at precision 6 (city-level) with full month breakdown and store it in a persistent table. This table feeds directly into:

- Tableau dashboards
- Python Folium maps
- Kepler.gl web visualizations
- DDC weekly situation report PDFs

## Query: Read the Pre-Built Table

In [9]:
run_query("""
SELECT
    report_month,
    cell_id,
    case_count,
    df_count,
    dhf_count,
    dss_count,
    dhf_count + dss_count AS severe_count,
    avg_age,
    ST_AsText(hex_geom) AS cell_polygon_wkt
FROM ddc_training.dengue_hexbin_report
ORDER BY report_month, case_count DESC
LIMIT 30
""")

,report_month,cell_id,case_count,df_count,dhf_count,dss_count,severe_count,avg_age,cell_polygon_wkt
0,2024-06-01,w4rw04,2,1,1,0,1,10.0,"POLYGON ((100.546875 13.7219238281, 100.557861..."
1,2024-06-01,w4rw01,1,1,0,0,0,25.0,"POLYGON ((100.546875 13.7164306641, 100.557861..."
2,2024-07-01,w4rw04,4,3,0,1,1,25.8,"POLYGON ((100.546875 13.7219238281, 100.557861..."
3,2024-07-01,w4rw23,2,1,1,0,1,16.0,"POLYGON ((100.557861328 13.7603759766, 100.568..."
4,2024-07-01,w5q6uk,2,1,1,0,1,15.5,"POLYGON ((98.9758300781 18.7866210938, 98.9868..."
5,2024-07-01,w4rw03,1,1,0,0,0,45.0,"POLYGON ((100.557861328 13.7164306641, 100.568..."
6,2024-07-01,w4rw24,1,1,0,0,0,38.0,"POLYGON ((100.546875 13.7658691406, 100.557861..."
7,2024-07-01,w5q6u7,1,1,0,0,0,35.0,"POLYGON ((98.9758300781 18.7811279297, 98.9868..."
8,2024-07-01,w4rw01,1,0,1,0,1,18.0,"POLYGON ((100.546875 13.7164306641, 100.557861..."
9,2024-08-01,w4rw23,3,2,0,1,1,20.7,"POLYGON ((100.557861328 13.7603759766, 100.568..."


,report_month,cell_id,case_count,df_count,dhf_count,dss_count,severe_count,avg_age,cell_polygon_wkt
0,2024-06-01,w4rw04,2,1,1,0,1,10.0,"POLYGON ((100.546875 13.7219238281, 100.557861..."
1,2024-06-01,w4rw01,1,1,0,0,0,25.0,"POLYGON ((100.546875 13.7164306641, 100.557861..."
2,2024-07-01,w4rw04,4,3,0,1,1,25.8,"POLYGON ((100.546875 13.7219238281, 100.557861..."
3,2024-07-01,w4rw23,2,1,1,0,1,16.0,"POLYGON ((100.557861328 13.7603759766, 100.568..."
4,2024-07-01,w5q6uk,2,1,1,0,1,15.5,"POLYGON ((98.9758300781 18.7866210938, 98.9868..."
5,2024-07-01,w4rw03,1,1,0,0,0,45.0,"POLYGON ((100.557861328 13.7164306641, 100.568..."
6,2024-07-01,w4rw24,1,1,0,0,0,38.0,"POLYGON ((100.546875 13.7658691406, 100.557861..."
7,2024-07-01,w5q6u7,1,1,0,0,0,35.0,"POLYGON ((98.9758300781 18.7811279297, 98.9868..."
8,2024-07-01,w4rw01,1,0,1,0,1,18.0,"POLYGON ((100.546875 13.7164306641, 100.557861..."
9,2024-08-01,w4rw23,3,2,0,1,1,20.7,"POLYGON ((100.557861328 13.7603759766, 100.568..."


**Note:** This table already exists in your database. It was created during data initialization. We're reading it here to verify its structure and sample data.

---

# Part 7: Interactive Heatmap Visualization

## Using the `show_heatmap()` Helper

The `show_heatmap()` function takes your query result and renders an interactive map with colored cells. Red cells = high case count, yellow cells = low count.

## Heatmap: City-Level Aggregation (Precision 6)

In [10]:
show_heatmap("""
SELECT
    ST_AsText(hex_geom) AS wkt,
    case_count
FROM ddc_training.dengue_hexbin_report
WHERE report_month = (SELECT MAX(report_month) FROM ddc_training.dengue_hexbin_report)
""", title="Dengue Heatmap (Precision 6) — Latest Month")

**Interpretation:**

- **Red cells** = highest case density (immediate intervention needed)
- **Orange cells** = elevated risk
- **Yellow cells** = emerging hotspots (monitor closely)
- **White cells** = no cases (safe zone)

This map directly informs DDC's weekly situation report and vector control deployments.

---

# Part 8: Cross-Module Integration — Hexbin + Risk Zones

## Bridging Module 2 and Module 3

Module 2 created **risk zones** (buffers around hospitals with severity scoring). Module 3 created **heatmap cells** (spatial aggregation of cases). Now we **intersect them** to answer:

**"Which high-risk zones (RED tier from Module 2) overlap with the hottest heatmap cells?"**

This drives **resource allocation**: DDC sends vector control teams to cells that are both (a) dense with cases AND (b) flagged as high-risk zones.

## Query: Heatmap Cells Overlapping with RED Risk Zones

In [ ]:
run_query("""
SELECT
    hr.report_month,
    hr.cell_id,
    hr.case_count AS cell_cases,
    rz.risk_tier,
    rz.province_name,
    rz.area_sq_km AS risk_zone_area,
    ROUND(100.0 * hr.case_count / NULLIF(rz.area_sq_km, 0), 2) AS cases_per_sq_km
FROM ddc_training.dengue_hexbin_report hr
JOIN ddc_training.dengue_risk_zones rz
    ON ST_Intersects(ST_GeomFromText(ST_AsText(rz.zone_geom)),
                     ST_GeomFromText(ST_AsText(hr.hex_geom)))
WHERE rz.risk_tier = 'RED'
ORDER BY hr.report_month, hr.case_count DESC
""")

**Key Findings:**

- High `case_count` + RED risk tier = **critical zone** (deploy mobile teams, spray operations)
- High `cases_per_sq_km` = intense hotspot (small area, many cases = harder to reach all houses)
- Low risk tier but rising case_count = **emerging threat** (monitor, prepare staging areas)

This cross-module query is used in the **daily DDC briefing** to prioritize field operations.

---

# Part 9: Key SQL Functions Reference

## Spatial Functions Introduced in Module 3

| Function | Input | Output | Purpose |
|----------|-------|--------|----------|
| `ST_GeoHash(geom)` | Point geometry | String (e.g., "s00tvu") | Encode point as grid cell code |
| `SUBSTR(hash, 1, N)` | String, position, length | Truncated string | Control cell size by precision |
| `ST_GeomFromGeoHash(hash)` | Hash string | Polygon geometry | Decode hash back to cell boundary |
| `ST_AsText(geom)` | Geometry | WKT string | Export geometry for visualization |
| `DATE_TRUNC('month', date)` | Timestamp | Timestamp (first of month) | Time binning for temporal aggregation |
| `ST_Intersects(geom1, geom2)` | Two geometries | Boolean | Check if geometries overlap |
| `ST_GeomFromText(wkt)` | WKT string | Geometry | Import WKT polygon into Vertica |

## Vertica Tip: Hexagons vs. Rectangles

Vertica Enterprise users can also use:
- `ST_HexagonCellId(point, res)` — native hexagonal grid (better aesthetics, fewer edge cases)
- `ST_Hexagon(hex_id)` — retrieve hexagon polygon

However, **GeoHash (rectangular cells)** is available in Vertica Community Edition and works equally well for heatmaps.

---

# Part 10: Challenge Exercise

## Try It Yourself: Precision 7 Heatmap

Now that you understand the pattern, try generating a **neighborhood-level heatmap** at precision 7. 

Your task:

1. Modify the query from **Part 3** ("Retrieve Cell Polygons for Visualization") to use precision 7 instead of precision 6
2. Run it and count how many unique cells appear
3. Identify the **single neighborhood with the highest case count**
4. Export that WKT polygon and paste it into [geojson.io](https://geojson.io) to visualize the exact neighborhood boundary

### Why This Matters

Precision 7 shows DDC's vector control teams exactly which neighborhoods to focus on. A team can cover a ~150m × 150m area in one day of house-to-house visits.

### Hints

- Change `1, 6` to `1, 7` in the `SUBSTR()` calls
- Everything else stays the same
- The query will run slightly slower (more cells to group, bigger result set)

In [ ]:
# Write your precision 7 query here:
run_query("""
SELECT
    cell_id,
    case_count,
    severe_count,
    ST_AsText(ST_GeomFromGeoHash(cell_id)) AS cell_polygon_wkt
FROM (
    SELECT
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 7) AS cell_id,
        COUNT(*) AS case_count,
        SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count
    FROM ddc_training.dengue_cases
    GROUP BY SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 7)
) bins
ORDER BY case_count DESC
""")

---

# Summary: Three Modules, One Workflow

Across Course 1, we've built a complete spatial surveillance pipeline:

## Module 0: Setup
- Created base tables: `dengue_cases`, `hospitals`, `schools`, `province_boundaries`

## Module 2: Risk Zones (Buffers & Severity)
- Created `dengue_risk_zones` — graduated buffers (5km, 10km) around hospitals with severity scoring
- Identified RED/ORANGE/YELLOW zones for resource allocation

## Module 3: Spatial Heatmaps (This Module)
- Created `dengue_hexbin_report` — GeoHash binning at multiple resolutions with temporal breakdown
- Generated visualizations showing outbreak hotspots and progression

## The Integration
- **Part 8** joined both tables to answer: **"Which hottest cells overlap with highest-risk zones?"**
- Result: DDC's weekly operational briefing with prioritized response zones

## Next Steps
- **SQL+Tableau analysts** can export `dengue_hexbin_report` to Tableau and build public dashboards
- **Python data scientists** can read the same table and build Folium/Kepler.gl web maps
- **GIS specialists** can load WKT polygons into QGIS for advanced spatial analysis

---

## DDC Training Project
*Practical Spatial Analytics for Disease Surveillance*